# The matrix exponential and the expected order flow

The answers to Section 10 of the sample questions, `documentation/tex/questions`, run.
The questions concern the forward means of Proposition 1.3.33 of the lecture notes,
the mean response of Corollary 1.3.35,
and the expected order flow of Section 1.4.3, on pairs of types read as market buys and
market sells, with pressure $\varpi = (+1, -1)$.

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
Where a stem gives a number, the number is checked first.

In [ ]:
import contextlib
import io
import math
import timeit

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams, mean_response_integral
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")

The three pairs of the section, as the table at its head gives them.

In [ ]:
PAIRS = {
    "self-exciting": config.self_exciting_pair_params(),
    "cross-exciting": config.cross_exciting_pair_params(),
    "one-way": config.one_way_pair_params(),
}
TABLE = {
    "self-exciting": ([0.25, 0.25], [[2.8, 0.2], [0.2, 2.8]], [1.0, 1.0]),
    "cross-exciting": ([0.25, 0.25], [[0.2, 2.8], [2.8, 0.2]], [1.0, 1.0]),
    "one-way": ([0.4, 0.1], [[0.8, 0.0], [4.0, 2.4]], [0.5, 1.5]),
}
for name, params in PAIRS.items():
    baseline, excitation, stationary = TABLE[name]
    assert np.allclose(params.baseline, baseline)
    assert np.allclose(params.excitation, excitation)
    assert params.decay == 4.0
    assert np.allclose(params.stationary_intensity(), stationary)
    print(f"{name:>15}: eigenvalues of K {np.sort(np.linalg.eigvals(params.hurwitz_matrix).real)}")

## `question.expmOfKTimesTheHorizon`: the exponential of $Kh$, of $K$ times $h$, and entry by entry

In [ ]:
for x, value in ((-0.5, 0.607), (-0.7, 0.497), (-1.0, 0.368), (-1.4, 0.247),
                 (-0.6, 0.549), (0.1, 1.105)):
    assert round(math.exp(x), 3) == value


def run():
    params = config.self_exciting_pair_params()
    K, h = params.hurwitz_matrix, 0.5
    for R in (expm(K * h), expm(K) * h, np.exp(K * h)):
        print(R[:, 0].round(3).tolist())


by_hand = [
    [(math.exp(-0.5) + math.exp(-0.7)) / 2, (math.exp(-0.5) - math.exp(-0.7)) / 2],
    [0.5 * (math.exp(-1.0) + math.exp(-1.4)) / 2, 0.5 * (math.exp(-1.0) - math.exp(-1.4)) / 2],
    [math.exp(-0.6), math.exp(0.1)],
]
print("by hand:", [np.round(column, 3).tolist() for column in by_hand])
lines = outcome(run).split("\n")
print(lines)
verify("question.expmOfKTimesTheHorizon", "C", [
    lines == ["[0.552, 0.055]"] * 3,
    lines == ["[0.552, 0.055]", "[0.552, 0.055]", "[0.549, 1.105]"],
    lines == ["[0.552, 0.055]", "[0.154, 0.03]", "[0.549, 1.105]"],
    lines == ["[0.552, 0.055]", "[0.154, 0.03]", "[0.552, 0.055]"],
])

## `question.seriesOnKAgainstSeriesOnTheExcitation`: cancellation in the series of $e^{Kh}$

The facts the stem gives: the entries of $e^{10K}$, the largest of $66^k/k!$, and $e^{30}$.

In [ ]:
cross = config.cross_exciting_pair_params()
assert np.allclose(np.sort(np.linalg.eigvals(cross.hurwitz_matrix).real), [-6.6, -1.0])
assert np.allclose(np.sort(np.linalg.eigvals(cross.excitation).real), [-2.6, 3.0])
assert np.allclose(expm(cross.hurwitz_matrix * 10.0), 2.27e-5, rtol=0.01)
largest = max(math.exp(k * math.log(66) - math.lgamma(k + 1)) for k in range(400))
print(f"largest of 66^k / k!: {largest:.2e}, e^30 = {math.exp(30):.2e}")
assert 1e27 < largest < 3e27 and 5e12 < math.exp(30) < 2e13


def run():
    def series_on_k(params, h):
        term, total = np.eye(2), np.eye(2)
        for k in range(1, 400):
            term = term @ (params.hurwitz_matrix * h) / k
            total = total + term
        return total

    def series_on_a(params, h):
        term, total = np.eye(2), np.eye(2)
        for k in range(1, 400):
            term = term @ (params.excitation * h) / k
            total = total + term
        return np.exp(-params.decay * h) * total

    params = config.cross_exciting_pair_params()
    for h in (1.0, 10.0):
        exact = expm(params.hurwitz_matrix * h)
        print(np.allclose(series_on_k(params, h), exact),
              np.allclose(series_on_a(params, h), exact))


lines = outcome(run).split("\n")
print(lines)
verify("question.seriesOnKAgainstSeriesOnTheExcitation", "B", [
    lines == ["True True", "True True"],
    lines == ["True True", "False True"],
    lines == ["True True", "True False"],
    lines == ["True True", "False False"],
    lines == ["False True", "False True"],
])

## `question.eigenvectorsTransposedForTheInverse`: $P^{-1}$ against $P^\top$

In [ ]:
one_way = config.one_way_pair_params()
kappa, P = np.linalg.eig(one_way.hurwitz_matrix)
print("one-way: eigenvalues", kappa.real, " eigenvectors (columns)\n", P.real)
assert np.allclose(one_way.hurwitz_matrix @ [0.0, 1.0], -1.6 * np.array([0.0, 1.0]))
assert np.allclose(one_way.hurwitz_matrix @ [0.4, -1.0], -3.2 * np.array([0.4, -1.0]))


def run():
    def by_inverse(params, h):
        kappa, P = np.linalg.eig(params.hurwitz_matrix)
        return (P * np.exp(kappa * h)) @ np.linalg.inv(P)

    def by_transpose(params, h):
        kappa, P = np.linalg.eig(params.hurwitz_matrix)
        return (P * np.exp(kappa * h)) @ P.T

    for params in (config.self_exciting_pair_params(),
                   config.one_way_pair_params()):
        exact = expm(params.hurwitz_matrix * 0.5)
        print(np.allclose(by_inverse(params, 0.5), exact),
              np.allclose(by_transpose(params, 0.5), exact))


lines = outcome(run).split("\n")
print(lines)
verify("question.eigenvectorsTransposedForTheInverse", "A", [
    lines == ["True True", "True False"],
    lines == ["True True", "True True"],
    lines == ["True False", "True False"],
    lines == ["True True", "False False"],
])

## `question.blockExponentialOfOneMarketBuy`: the blocks of $e^{Gh}$ for one type

The distractors are the counts of three misreadings of the blocks:
$K$ taken for zero ($\Phi = h$, $\int\Phi = h^2/2$),
the entry $(0, 2)$ taken for zero as in the generator,
and the first block, $e^{Kh}$, taken for $\Phi(h)$.

In [ ]:
def run():
    mu, a, z, h = 0.5, 2.0, 1.0, 0.5
    generator = np.array([
        [a - 4.0, 1.0, 0.0],
        [0.0, 0.0, 1.0],
        [0.0, 0.0, 0.0],
    ])
    blocks = expm(generator * h)
    print(blocks[1].round(3).tolist())
    print(round(h * mu + a * blocks[0, 1] * z + a * blocks[0, 2] * mu, 3))


mu, a, z, h = 0.5, 2.0, 1.0, 0.5
exponential, integral = math.exp(-2 * h), (1 - math.exp(-2 * h)) / 2
double = h / 2 - (1 - math.exp(-2 * h)) / 4
package = HawkesParams(np.array([mu]), np.array([[a]]), 4.0).forward_mean_counts(np.array([z]), h)
print("by hand: e^{Kh}, Phi, int Phi =", round(exponential, 3), round(integral, 3), round(double, 3))
print("the package's expected count:", package.round(3).tolist())
print("K taken for zero:", round(h * mu + a * h * z + a * h ** 2 / 2 * mu, 3),
      " entry (0, 2) zero:", round(h * mu + a * integral * z, 3),
      " e^{Kh} for Phi:", round(h * mu + a * exponential * z + a * double * mu, 3))
row, count = outcome(run).split("\n")
print(row, "|", count)
verify("question.blockExponentialOfOneMarketBuy", "D", [
    row == "[0.0, 0.0, 0.5]" and count == "0.974",
    row == "[0.0, 1.0, 0.5]" and count == "1.375",
    row == "[0.0, 1.0, 0.5]" and count == "0.882",
    row == "[0.0, 1.0, 0.5]" and count == "0.974",
    row == "[0.0, 1.0, 0.5]" and count == "1.078",
])

## `question.integralOfTheResponseAtTheCriticalPoint`: $\Phi(h)$ at $\rho = 1$

In [ ]:
critical = np.array([[2.0, 2.0], [2.0, 2.0]])
print("eigenvalues of Gamma:", np.linalg.eigvals(critical / 4.0).real)
assert round(math.exp(-2.0), 3) == 0.135
print("Phi(0.5) by hand:", round((0.5 + (1 - math.exp(-2)) / 4) / 2, 3),
      round((0.5 - (1 - math.exp(-2)) / 4) / 2, 3),
      "  pseudo-inverse by hand:", round((1 - math.exp(-2)) / 8, 3))

In [ ]:
def phi(excitation, decay, h):
    K = excitation - decay * np.eye(2)
    return np.linalg.solve(K, expm(K * h) - np.eye(2))

phi_a = phi

In [ ]:
def phi(excitation, decay, h):
    K = excitation - decay * np.eye(2)
    return np.linalg.pinv(K) @ (expm(K * h) - np.eye(2))

phi_b = phi

In [ ]:
phi = mean_response_integral
phi_c = phi

In [ ]:
def run(phi):
    excitation = np.array([[2.0, 2.0], [2.0, 2.0]])
    print(phi(excitation, 4.0, 0.5).round(3).tolist())


a, b, c = outcome(run, phi_a), outcome(run, phi_b), outcome(run, phi_c)
print("A:", a, "   B:", b, "   C:", c)
blocks_value = "[[0.358, 0.142], [0.142, 0.358]]"
pseudo_value = "[[0.108, -0.108], [-0.108, 0.108]]"
verify("question.integralOfTheResponseAtTheCriticalPoint", "C", [
    a == "LinAlgError" and b == c == blocks_value,
    a == b == "LinAlgError" and c == blocks_value,
    a == "LinAlgError" and b == pseudo_value and c == blocks_value,
    a == "LinAlgError" and b == pseudo_value and c == "LinAlgError",
])

## `question.oneMoreMarketBuyDisplacesTheSells`: a column, a row, and a mean state

In [ ]:
one_way = config.one_way_pair_params()
print("e^{K/2} =", one_way.mean_response(0.5).round(3).tolist())
assert one_way.mean_response(0.5).round(3).tolist() == [[0.202, 0.0], [0.619, 0.449]]
m = one_way.stationary_mean()
assert np.allclose(m, [0.125, 0.375])
exponential = np.array([[0.202, 0.0], [0.619, 0.449]])
print("m + e^{Kh} (e_1 - m) from the three-digit exponential:",
      (m + exponential @ (np.array([1.0, 0.0]) - m)).round(3).tolist())


def run():
    params = config.one_way_pair_params()
    z, buy, h = np.array([0.3, 0.9]), np.array([1.0, 0.0]), 0.5
    mean_state = params.forward_mean_state
    a = mean_state(z + buy, h) - mean_state(z, h)
    b = params.mean_response(h)[0]
    c = params.forward_mean_state(buy, h)
    for displacement in (a, b, c):
        print(displacement.round(3).tolist())


lines = outcome(run).split("\n")
print(lines)
verify("question.oneMoreMarketBuyDisplacesTheSells", "A", [
    lines == ["[0.202, 0.619]", "[0.202, 0.0]", "[0.302, 0.748]"],
    lines == ["[0.202, 0.619]", "[0.202, 0.619]", "[0.302, 0.748]"],
    lines == ["[0.202, 0.619]", "[0.202, 0.0]", "[0.202, 0.619]"],
    lines == ["[0.202, 0.619]"] * 3,
])

## `question.peakOfTheTotalMeanResponse`: column sums against row sums on a clock

In [ ]:
assert round(math.log(1.2), 3) == 0.182 and round(math.log(10 / 7), 3) == 0.357
horizons = np.linspace(0.0, 1.0, 11)
closed_form = np.array([
    [[math.exp(-3.2 * h), 0.0],
     [2.5 * (math.exp(-1.6 * h) - math.exp(-3.2 * h)), math.exp(-1.6 * h)]]
    for h in horizons
])
assert np.allclose(one_way.mean_response(horizons), closed_form)
print("peaks by hand:", round(math.log(1.2) / 1.6, 3), round(math.log(10 / 7) / 1.6, 3))


def run():
    params = config.one_way_pair_params()
    clock = np.linspace(0.0, 1.0, 1001)
    R = params.mean_response(clock)
    for total in (R.sum(axis=1), R.sum(axis=2)):
        print(clock[total.argmax(axis=0)].round(3).tolist())


lines = outcome(run).split("\n")
print(lines)
verify("question.peakOfTheTotalMeanResponse", "B", [
    lines == ["[0.0, 0.223]", "[0.114, 0.0]"],
    lines == ["[0.114, 0.0]", "[0.0, 0.223]"],
    lines == ["[0.0, 0.0]", "[0.0, 0.0]"],
    lines == ["[0.114, 0.0]", "[0.114, 0.0]"],
])

## `question.responseIntegralCountsStrictDescendants`: column sums of $(I - \Gamma)^{-1}$

In [ ]:
assert np.abs(expm(one_way.hurwitz_matrix * 50.0)).max() < 1e-30
assert np.allclose(np.linalg.inv(np.eye(2) - one_way.branching_matrix), [[1.25, 0.0], [3.125, 2.5]])


def run():
    params = config.one_way_pair_params()
    ones, gamma = np.ones(2), params.branching_matrix
    inverse = np.linalg.inv(np.eye(2) - gamma)
    a = ones @ params.excitation @ params.mean_response_integral(50.0)
    b = ones @ inverse
    c = inverse @ ones - 1
    for counts in (a, b, c):
        print(counts.round(3).tolist())


lines = outcome(run).split("\n")
print(lines)
verify("question.responseIntegralCountsStrictDescendants", "D", [
    lines == ["[3.375, 1.5]", "[4.375, 2.5]", "[3.375, 1.5]"],
    lines == ["[4.375, 2.5]", "[4.375, 2.5]", "[0.25, 4.625]"],
    lines == ["[0.25, 4.625]", "[1.25, 5.625]", "[0.25, 4.625]"],
    lines == ["[3.375, 1.5]", "[4.375, 2.5]", "[0.25, 4.625]"],
])

## `question.signedCountDriftOnTheOneWayPair`: the baseline terms of a pair that is not direction-symmetric

In [ ]:
assert np.abs(expm(one_way.hurwitz_matrix * 100.0)).max() < 1e-60
progeny = one_way.branching_matrix @ np.linalg.inv(np.eye(2) - one_way.branching_matrix)
print("varpi' Gamma (I - Gamma)^-1 =", (np.array([1.0, -1.0]) @ progeny).tolist(),
      " at m:", np.array([1.0, -1.0]) @ progeny @ one_way.stationary_mean())


def run():
    params = config.one_way_pair_params()
    pressure, z = np.array([1.0, -1.0]), params.stationary_mean()
    terms = params.forward_mean_counts_terms(z, 100.0).contracted(pressure)
    print(round(terms.total, 2), round(terms.descendants_of_state, 2))


printed = outcome(run)
print(printed)
verify("question.signedCountDriftOnTheOneWayPair", "E", [
    printed == "-0.92 -0.92",
    printed == "0.0 0.0",
    printed == "-100.0 0.0",
    printed == "-100.92 -0.92",
    printed == "-100.0 -0.92",
])

## `question.signedDescendantsOfTheStateContracted`: $A$ commutes with $\Phi(h)$, and a transposed contraction

In [ ]:
def run():
    params = config.one_way_pair_params()
    pressure, z, h = np.array([1.0, -1.0]), np.array([0.3, 0.9]), 0.5
    A, phi = params.excitation, params.mean_response_integral(h)
    terms = params.forward_mean_counts_terms(z, h).contracted(pressure)
    reference = terms.descendants_of_state
    candidates = (
        pressure @ A @ phi @ z,
        pressure @ phi @ A @ z,
        z @ A @ phi @ pressure,
        pressure @ A.T @ phi @ z,
    )
    print([bool(np.isclose(x, reference)) for x in candidates])


printed = outcome(run)
print(printed)
verify("question.signedDescendantsOfTheStateContracted", "C", [
    printed == "[True, False, False, False]",
    printed == "[True, True, True, False]",
    printed == "[True, True, False, False]",
    printed == "[True, False, True, False]",
    printed == "[True, True, True, True]",
])

## `question.regimeSignAfterABurstOfBuys`: continuation and reversal over half a second

In [ ]:
assert round(math.exp(-0.7), 3) == 0.497 and round(math.exp(-3.3), 3) == 0.037
for theta in (2.6, -2.6):
    print(f"theta = {theta:+}: theta (1 - e^((theta - 4) h)) / (4 - theta) (z1 - z2) =",
          round(theta * (1 - math.exp((theta - 4) * 0.5)) / (4 - theta) * 1.5, 3))


def run():
    pressure, z, h = np.array([1.0, -1.0]), np.array([2.0, 0.5]), 0.5
    for params in (config.self_exciting_pair_params(),
                   config.cross_exciting_pair_params()):
        terms = params.forward_mean_counts_terms(z, h).contracted(pressure)
        drift = h * pressure @ params.excitation @ z
        print(round(terms.total, 3), round(drift, 3))


lines = outcome(run).split("\n")
print(lines)
verify("question.regimeSignAfterABurstOfBuys", "D", [
    lines == ["-0.569 -1.95", "1.402 1.95"],
    lines == ["1.95 1.95", "-1.95 -1.95"],
    lines == ["1.402 1.95", "-1.402 -1.95"],
    lines == ["1.402 1.95", "-0.569 -1.95"],
])

## `question.meanResponseOnAClockByEigenvalues`: one `expm` per horizon against one eigendecomposition

Both are timed on the one-way pair, on clocks of $10^4$ and $10^5$ horizons, and checked to
agree.  The choices are read off the ratio of the times at each size and off the growth of
each time from $10^4$ to $10^5$.

In [ ]:
def responses_a(K, clock):
    return np.array([expm(K * h) for h in clock])

In [ ]:
def responses_b(K, clock):
    kappa, P = np.linalg.eig(K)
    growth = np.exp(np.multiply.outer(clock, kappa))
    return ((P * growth[:, None, :]) @ np.linalg.inv(P)).real

In [ ]:
def seconds(responses, n, calls):
    return timeit.timeit(lambda: responses(K, clocks[n]), number=calls) / calls


K = config.one_way_pair_params().hurwitz_matrix
clocks = {n: np.linspace(0.0, 3.0, n) for n in (10_000, 100_000)}
for clock in clocks.values():
    assert np.allclose(responses_a(K, clock), responses_b(K, clock))

# Seven rounds of the four timings back to back.  A ratio taken within a round, and its
# median over the rounds, hold steady under a load that moves the times themselves.  B is
# called 200_000 // n times a timing, so that its two timings last about as long.
rounds = [
    {(name, n): seconds(responses, n, 1 if name == "A" else 200_000 // n)
     for name, responses in (("A", responses_a), ("B", responses_b))
     for n in clocks}
    for _ in range(7)
]
median = lambda values: float(np.median(values))
for n in clocks:
    a, b = median([r["A", n] for r in rounds]), median([r["B", n] for r in rounds])
    print(f"n = {n:>6}:  A {a * 1e3:8.2f} ms   B {b * 1e3:8.3f} ms")

ratio = {n: median([r["A", n] / r["B", n] for r in rounds]) for n in clocks}
growth_a = median([r["A", 100_000] / r["A", 10_000] for r in rounds])
growth_b = median([r["B", 100_000] / r["B", 10_000] for r in rounds])
print(f"A over B {ratio[10_000]:.1f} and {ratio[100_000]:.1f}; "
      f"growth of A {growth_a:.1f}x, growth of B {growth_b:.1f}x")
verify("question.meanResponseOnAClockByEigenvalues", "B", [
    all(r < 1 for r in ratio.values()),
    all(r > 5 for r in ratio.values()) and 4 < growth_a < 25 and 4 < growth_b < 25,
    all(r > 5 for r in ratio.values()) and growth_b < 2,
    all(0.5 < r < 2 for r in ratio.values()),
])